# DAEN 328 Assignment 5: Load and Serve

**Name:** Anna Williamson

Save this notebook as `A5_LastName_FirstName.ipynb`. It must run **top to bottom** (Kernel > Restart & Run All) with all outputs showing when you submit.

Work through `Assignment05_Guide.ipynb` first. Every technique you need is taught there on practice data. This notebook is where you apply it to `EmployeeDB`.

Your dashboard (Part C) is a separate file, `app.py`. See the instructions PDF.

## CONFIG: edit this cell once

In [ ]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine, text

DB_NAME = "EmployeeDB"
DB_USER = "postgres"
DB_PASSWORD = "YOUR_PASSWORD"      # change to YOUR PostgreSQL password
DB_HOST = "localhost"
DB_PORT = "5432"

CSV_FILE = "Cleaned_data.csv"   # your Assignment 3 output, in the same folder as this notebook

DB_PARAMS = {"dbname": DB_NAME, "user": DB_USER, "password": DB_PASSWORD,
             "host": DB_HOST, "port": DB_PORT}
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# Part A: Load your cleaned CSV into the 7 EmployeeDB tables

### A1. Read and inspect

Read the CSV with every column as text. Show its shape and the number of blanks per column.

In [2]:
data = pd.read_csv("Cleaned_data.csv", dtype=str)
print(data.shape)
data.isna().sum()

(14321, 24)


EmployeeID                0
Name                      0
HighestDegree             0
Division                  0
DivisionAddress           0
DivisionPhone            64
DivStreetAddress          0
DivCity                   0
DivState                  0
DivZip                    0
PersonalAddress           0
PerStreetAddress          0
PerCity                   0
PerState                  0
PerZip                    0
Email                     0
PhoneNumber               0
Department                0
DepartmentDescription    54
JobTitle                  1
JobDescription            1
Salary                    0
Birthdate                 0
SocialSecurity            0
dtype: int64

### A2. Clean and validate in pandas

Required:
- Strip spaces from text columns.
- Convert `Birthdate` to a real date (`pd.to_datetime(..., errors="coerce").dt.date`) and **print how many could not be read**.
- Convert `Salary` and `EmployeeID` to numbers.
- Put every row that cannot satisfy your schema's NOT NULL or CHECK constraints into a `rejects` DataFrame, save it as `rejects_employees.csv`, and print rows read / good / rejected. Never invent a value (such as a salary of 0) to get a row past a constraint.

In [3]:
df = data.copy()

# Strip spaces from every text column
for col in df.columns:
    df[col] = df[col].str.strip()

df["Birthdate"] = pd.to_datetime(df["Birthdate"], format="%m/%d/%Y", errors="coerce").dt.date
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")
df["EmployeeID"] = pd.to_numeric(df["EmployeeID"], errors="coerce").astype("Int64")

print("Birthdates that are unreadable:", df["Birthdate"].isna().sum())
print("Salaries that are unreadable:", df["Salary"].isna().sum())
print("EmployeeIDs that are unreadable:", df["EmployeeID"].isna().sum())

Birthdates that are unreadable: 0
Salaries that are unreadable: 0
EmployeeIDs that are unreadable: 0


In [4]:
# insert honorific column if doesn't already exist
df["Honorific"] = df.get("Honorific")
df.head()

,EmployeeID,Name,HighestDegree,Division,DivisionAddress,DivisionPhone,DivStreetAddress,DivCity,DivState,DivZip,...,Email,PhoneNumber,Department,DepartmentDescription,JobTitle,JobDescription,Salary,Birthdate,SocialSecurity,Honorific
0,436822,Steven Sean Johnson,MA,South Division,456 South Ave Austin TX 73301,(512)555-6789,456 South Ave,Austin,TX,73301,...,steven_johnson_145@KLB.com,(983)309-3096,Administrative,"Manage, coordinate, and facilitate internal co...",Secretary Level 2,Provides administrative support within the Sou...,43996.06,1971-12-10,149-51-0794,None
1,224873,Robert Matthew Schultz,CTE,North Division,123 North St Springfield IL 62701,(217)555-1234,123 North St,Springfield,IL,62701,...,robert_schultz_117@KLB.com,(696)415-7443,Operations,"Manage all purchasing, manufacturing, and logi...",Operations Associate Level 4,Performs production and operational tasks with...,37820.64,2005-01-18,097-83-2017,None
2,494157,Donna Brittany Ford,AB,North Division,123 North St Springfield IL 62701,(217)555-1234,123 North St,Springfield,IL,62701,...,donna_ford_151@KLB.com,(217)555-1234,Administrative,"Manage, coordinate, and facilitate internal co...",Secretary Level 5,Provides administrative support within the Nor...,47606.65,1965-02-13,494-41-0582,None
3,460666,Teresa Jennifer Hernandez,THD,South Division,456 South Ave Austin TX 73301,(512)555-6789,456 South Ave,Austin,TX,73301,...,teresa_hernandez_162@KLB.com,(894)497-2204,Administrative,"Manage, coordinate, and facilitate internal co...",Secretary Level 5,Provides administrative support within the Sou...,41088.33,1975-05-21,421-73-2111,None
4,471359,Christopher Perry Lopez,AB,West Division,101 West Dr Sacramento CA 94203,(916)555-5678,101 West Dr,Sacramento,CA,94203,...,christopher_lopez_100@KLB.com,(859)491-9776,Administrative,"Manage, coordinate, and facilitate internal co...",Secretary Level 6,Provides administrative support within the Wes...,45006.00,1973-10-01,343-92-4420,None


In [5]:
# identify job title(s) causing loading issue
titles_length = df['JobTitle'].str.len()
print(df.loc[titles_length == titles_length.max(), 'JobTitle'].unique())
print(f"Max job title length: {titles_length.max()}")

<ArrowStringArray>
['Human Resources Manager Level 1', 'Human Resources Manager Level 2',
 'Human Resources Manager Level 3']
Length: 3, dtype: str
Max job title length: 31.0


In [6]:
import re

# replace 'Human Resources' with 'HR'
def replace_hr(title):
    if not pd.isna(title):
        return re.sub("Human Resources", "HR", title)
    return title

df['JobTitle'] = df['JobTitle'].apply(replace_hr)

# check that new longest title is within character limit of 30
titles_length = df['JobTitle'].str.len()
print(df.loc[titles_length == titles_length.max(), 'JobTitle'].unique())
print(f"Max job title length: {titles_length.max()}")

<ArrowStringArray>
['Operations Associate Level 10']
Length: 1, dtype: str
Max job title length: 29.0


In [7]:
# need to check for NA values in non-nullable columns
# also need to validate salary constraint
reject_values = df['SocialSecurity'].isna() | df['Name'].isna() | df['HighestDegree'].isna() | df['Email'].isna() \
    | df['PhoneNumber'].isna() | df['Birthdate'].isna() | df['PerStreetAddress'].isna() | df['PerCity'].isna() \
    | df['PerState'].isna() | df['PerZip'].isna() | df['Department'].isna() | df["JobTitle"].isna() | df['Division'].isna() \
    | df['DivStreetAddress'].isna() | df['DivCity'].isna() | df['DivState'].isna() | df['DivZip'].isna() \
    | df["JobDescription"].isna() | df['Salary'].isna() | (df["Salary"] <= 0)

# move rejects to separate file
rejects = df[reject_values]
valid = df[~reject_values].copy()

rejects.to_csv("rejects_employees.csv", index=False)
print(f"Rows read: {len(df)}   good: {len(valid)}   rejected: {len(rejects)}")
rejects

Rows read: 14321   good: 14320   rejected: 1


,EmployeeID,Name,HighestDegree,Division,DivisionAddress,DivisionPhone,DivStreetAddress,DivCity,DivState,DivZip,...,Email,PhoneNumber,Department,DepartmentDescription,JobTitle,JobDescription,Salary,Birthdate,SocialSecurity,Honorific
9043,221765,Brittany Monica Sanders,HS,Central Division,202 Main St Columbus OH 43085,(916)555-6789,202 Main St,Columbus,OH,43085,...,brittany_sanders_177@KLB.com,(439)586-6814,Operations,"Manage all purchasing, manufacturing, and logi...",NaN,NaN,29304.32,1973-09-04,673-43-5379,None


### Given: `clean()` (run it, do not change it)

Pass **every** value through `clean()` when you insert it. pandas marks a blank as `NaN`, and if a `NaN` reaches PostgreSQL it is stored as the value `NaN`, not as NULL. A `NaN` salary passes `NOT NULL` and `CHECK (salary > 0)` with no error, and then every average that includes it comes out as `NaN`. `clean()` turns blanks into `None` (a real NULL), so the database rejects what it should. It also turns numpy numbers into plain Python numbers, which avoids the error `can't adapt type numpy.int64`.

In [8]:
def clean(value):
    # Turn pandas blanks into None (SQL NULL) and numpy numbers into plain Python numbers.
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value

### Given: start over with empty tables

While you build the loader, a bad attempt can leave wrong rows behind. To empty all 7 tables and restart the id numbers from 1, set `RESET = True`, run the cell, then set it back to `False`. This keeps your tables and constraints; it only removes the rows. Do not use `DROP TABLE`, which deletes your Assignment 4 schema.

Leave `RESET = False` when you submit.

In [ ]:
RESET = False

if RESET:
    with engine.begin() as conn:
        conn.execute(text("""
            TRUNCATE employeedetails, personaldetails, jobdescriptions, divisionlocations,
                     departments, jobtitles, divisions
            RESTART IDENTITY CASCADE;
        """))
    print("All 7 tables emptied.")

All 7 tables emptied.


### A3. The loader

Write `load_employees(good)`. Requirements:
- Loads all 7 tables, lookup tables first, `employeedetails` last, using the CSV-to-column map in the instructions PDF.
- Runs in **one transaction** (all or nothing).
- **Safe to re-run**: a second run adds zero rows to every table.
- Uses `%s` placeholders for every value, and passes every value through `clean()`.

In [10]:
def load_employees(good):
    conn = psycopg2.connect(**DB_PARAMS)
    try:
        with conn:
            with conn.cursor() as cur:
    
                # lookup tables 
                departments = good[["Department", "DepartmentDescription"]].drop_duplicates("Department")
                for _, row in departments.iterrows():
                    cur.execute(
                        "INSERT INTO public.departments (departmentname, departmentdescription) VALUES (%s, %s) "
                        "ON CONFLICT (departmentname) DO NOTHING;",
                        (clean(row["Department"]), clean(row["DepartmentDescription"])))

                for job_title in good["JobTitle"].unique():
                    cur.execute(
                        "INSERT INTO public.jobtitles (jobtitle) VALUES (%s) "
                        "ON CONFLICT (jobtitle) DO NOTHING;",
                        (clean(job_title),))

                for division in good["Division"].unique():
                    cur.execute(
                        "INSERT INTO public.divisions (divtype) VALUES (%s) "
                        "ON CONFLICT (divtype) DO NOTHING;",
                        (clean(division),))

                cur.execute("SELECT departmentname, departmentid FROM public.departments;")
                department_ids = dict(cur.fetchall())
                cur.execute("SELECT jobtitle, jobtitleid FROM public.jobtitles;")
                job_title_ids = dict(cur.fetchall())
                cur.execute("SELECT divtype, divisionid FROM public.divisions;")
                division_ids = dict(cur.fetchall())

                # key comes from csv
                people = good[["SocialSecurity", "Name", "Honorific", "HighestDegree", "Email", "PhoneNumber",
                                  "Birthdate", "PerStreetAddress", "PerCity", "PerState", "PerZip"]] \
                                .drop_duplicates("SocialSecurity")
                for _, row in people.iterrows():
                    cur.execute(
                        "INSERT INTO public.personaldetails (ssn, name, honorific, highestdegree, email, phonenumber, " \
                        "birthdate, perstreet, percity, perstate, perzip) "
                        "VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s) ON CONFLICT (ssn) DO NOTHING;",
                        (clean(row["SocialSecurity"]), clean(row["Name"]), clean(row["Honorific"]), 
                        clean(row["HighestDegree"]), clean(row["Email"]), clean(row["PhoneNumber"]), 
                        clean(row["Birthdate"]), clean(row["PerStreetAddress"]), clean(row["PerCity"]),
                        clean(row["PerState"]), clean(row["PerZip"])))

                # child of lookup table
                locations = good[["Division", "DivStreetAddress", "DivCity", "DivState", "DivZip", "DivisionPhone"]] \
                                .drop_duplicates("Division")
                for _, row in locations.iterrows():
                    div_id = division_ids[row["Division"]]
                    cur.execute(
                        '''INSERT INTO public.divisionlocations (divisionid, divstreet, divcity, divstate, divzip, divisionphone)
                           SELECT %s, %s, %s, %s, %s, %s
                           WHERE NOT EXISTS (SELECT 1 FROM public.divisionlocations
                                             WHERE divisionid = %s);''',
                        (div_id, clean(row["DivStreetAddress"]), clean(row["DivCity"]), clean(row["DivState"]),
                        clean(row["DivZip"]), clean(row["DivisionPhone"]), div_id))

                descriptions = good[["JobTitle", "Division", "JobDescription"]].drop_duplicates(["JobTitle", "Division"])
                for _, row in descriptions.iterrows():
                    job_id = job_title_ids[row["JobTitle"]]
                    div_id = division_ids[row["Division"]]
                    cur.execute(
                        '''INSERT INTO public.jobdescriptions (jobtitleid, divisionid, description)
                           SELECT %s, %s, %s
                           WHERE NOT EXISTS (SELECT 1 FROM public.jobdescriptions
                                             WHERE jobtitleid = %s and divisionid = %s);''',
                        (job_id, div_id, clean(row["JobDescription"]), job_id, div_id))

                # fact table
                for _, row in good.iterrows():
                    cur.execute(
                        '''INSERT INTO public.employeedetails
                           (employeeid, ssn, departmentid, jobtitleid, divisionid, salary)
                           SELECT %s, %s, %s, %s, %s, %s
                           WHERE NOT EXISTS (SELECT 1 FROM public.employeedetails
                                             WHERE employeeid = %s and ssn = %s);''',
                        (clean(row["EmployeeID"]), clean(row["SocialSecurity"]),
                         department_ids[row["Department"]], job_title_ids[row["JobTitle"]],
                         division_ids[row["Division"]], clean(row["Salary"]), clean(row["EmployeeID"]),
                         clean(row["SocialSecurity"])))
                
        print("Load finished and committed.")
    finally:
        conn.close()

### A4. Run it twice and reconcile

Run your loader, record the 7 table counts, run it **again**, record them again. Show one table with, for every table: the count you expected from the CSV, the count after run 1, the count after run 2, and whether all three agree.

Then run the given NaN check below it. Every count must be 0.

In [11]:
TABLES = ["personaldetails", "departments", "jobtitles", "divisions", "divisionlocations", "jobdescriptions", "employeedetails"]

def table_counts(schema, tables):
    with engine.connect() as conn:
        return {t: conn.execute(text(f"SELECT COUNT(*) FROM {schema}.{t};")).scalar() for t in tables}

load_employees(valid)
after_run_1 = table_counts("public", TABLES)

load_employees(valid)
after_run_2 = table_counts("public", TABLES)

expected_counts = {
    "personaldetails": valid["SocialSecurity"].nunique(),
    "departments": valid["Department"].nunique(),
    "jobtitles": valid["JobTitle"].nunique(),
    "divisions": valid["Division"].nunique(),
    "divisionlocations": valid["Division"].nunique(), # one location per division
    "jobdescriptions": valid[["JobTitle","Division"]].apply(tuple, axis=1).nunique(), # one description per combo
    "employeedetails": len(valid), # one employee per good CSV row
}

run_results = pd.DataFrame({"expected_from_csv": expected_counts,
                      "after_run_1": after_run_1,
                      "after_run_2": after_run_2})
run_results["ok"] = (run_results["expected_from_csv"] == run_results["after_run_1"]) & \
              (run_results["after_run_1"] == run_results["after_run_2"])
run_results

Load finished and committed.
Load finished and committed.


,expected_from_csv,after_run_1,after_run_2,ok
personaldetails,14320,14320,14320,True
departments,6,6,6,True
jobtitles,51,51,51,True
divisions,5,5,5,True
divisionlocations,5,5,5,True
jobdescriptions,235,235,235,True
employeedetails,14320,14320,14320,True


In [12]:
# Given: NaN check. Every count must be 0. If not, a value reached the database without clean().
with engine.connect() as conn:
    nan_check = {
        "salary = NaN":    conn.execute(text("SELECT COUNT(*) FROM employeedetails WHERE salary = 'NaN';")).scalar(),
        "name = 'NaN'":    conn.execute(text("SELECT COUNT(*) FROM personaldetails WHERE name = 'NaN';")).scalar(),
        "jobtitle = 'NaN'": conn.execute(text("SELECT COUNT(*) FROM jobtitles WHERE jobtitle = 'NaN';")).scalar(),
    }
nan_check

{'salary = NaN': 0, "name = 'NaN'": 0, "jobtitle = 'NaN'": 0}

### A5. Load report (write in this cell, 4 to 6 sentences)

- How many rows you read, loaded and rejected, and the reason for each kind of reject.
- Which loading pattern you used for `divisionlocations`, `jobdescriptions` and `employeedetails`, and why.
- One real problem you hit while building the loader (the error message or wrong count) and how you found and fixed it.

I read a total of 14321 rows, rejecting 1 row and loading the remaining 14320. The reason for the one row being rejected was because it had an NA value for both JobTitle and JobDescription, both of which are non-nullable in my schema. For the `divisionlocations`, `jobdescriptions`, and `employeedetails` tables, I used `SELECT ... WHERE NOT EXISTS ...` to skip locations/descriptions/employees that already exist in the database since there is no UNIQUE column from the csv for ON CONFLICT to check.

One issue I hit when building the loader was an issue involving the jobtitle column. In the last assignment, the instructions had specified maximum string lengths for each column to guide limits for VARCHAR, including that the longest job title was 29 characters, so my schema uses 30 characters as the VARCHAR limit for jobtitle. However, I ran into a `StringDataRightTruncation: value too long for type character varying(30)` error when running my loader that I traced back to job titles that were 31 characters long. This was due to the fact that my dataset used 'Human Resources' instead of 'HR' in job titles (Assignment 3 specified that either version was valid), making 'Human Resources Manager Level _' above the 30 character limit. To address this issue, I had to substitute all instances of 'Human Resources' in job titles with 'HR', after which I could successfully load my data into the database.

# Part B: SQL

Write your own SQL for each question in the code cell under it, using the `q()` helper, and leave the result showing. Rules for every question:
- Filter by **names** (`'Engineering'`, `'Austin'`), never by an id number you looked up.
- Never type a number that the database can compute (an average, a count).
- Show names, not ids, wherever a person would read the result.

In [13]:
def q(sql, params=None):
    return pd.read_sql(text(sql), engine, params=params)

**Q1.** Headcount by department: each department's name and number of employees, largest first.

In [14]:
# Q1
q('''
SELECT d.departmentname, COUNT(e.emp_id)
FROM public.employeedetails AS e
JOIN public.departments AS d ON e.departmentid = d.departmentid
GROUP BY d.departmentname
ORDER BY COUNT(e.emp_id) DESC;
''')

,departmentname,count
0,Operations,10580
1,Administrative,2000
2,Engineering,1085
3,Human Resources,645
4,Management,5
5,Executive,5


**Q2.** Engineering employees earning less than $100,000: name, job title and salary, highest salary first.

In [15]:
# Q2
q('''
SELECT p.name, t.jobtitle, e.salary
FROM public.personaldetails AS p
JOIN public.employeedetails AS e ON p.ssn = e.ssn
JOIN public.jobtitles AS t ON t.jobtitleid = e.jobtitleid
JOIN public.departments AS d ON d.departmentid = e.departmentid
WHERE e.salary < 100000 AND d.departmentname = 'Engineering'
ORDER BY e.salary DESC;
''')

,name,jobtitle,salary
0,Jennifer Bonnie Duffy,Engineer Level 4,99980.47
1,Luis Christopher Holder,Engineer Level 3,99890.86
2,Michelle Mary Davis,Engineer Level 1,99860.47
3,Lisa Crystal Austin,Engineer Level 3,99838.28
4,Catherine Deanna Hunter,Engineer Level 4,99820.89
...,...,...,...
391,Brian Shawn Parrish,Engineer Level 1,81480.18
392,Billy Daniel Jenkins,Engineer Level 1,81478.14
393,Kaitlyn Amanda Allen,Engineer Level 1,81471.96
394,Jeremiah Jared Marshall,Engineer Level 1,81424.51


**Q3.** Human Resources employees who work at the division located in **Austin**: name, job title and division. Find the division through its city; do not type the division's name.

In [16]:
# Q3
q('''
SELECT p.name, t.jobtitle, v.divtype
FROM public.personaldetails AS p
JOIN public.employeedetails AS e ON p.ssn = e.ssn
JOIN public.jobtitles AS t ON t.jobtitleid = e.jobtitleid
JOIN public.divisions AS v ON v.divisionid = e.divisionid
JOIN public.divisionlocations AS l ON l.divisionid = v.divisionid
JOIN public.departments AS d ON d.departmentid = e.departmentid
WHERE d.departmentname = 'Human Resources' AND l.divcity = 'Austin';
''')

,name,jobtitle,divtype
0,Daniel Nicholas Parker,HR Specialist 1,South Division
1,Nicole Carly Hicks,HR Specialist 1,South Division
2,Laura Amanda Gonzalez,HR Specialist 4,South Division
3,Morgan Katelyn Jacobson,HR Specialist 1,South Division
4,Raymond Andrew Wade,HR Manager Level 1,South Division
...,...,...,...
124,Kenneth Jeff Stout,HR Specialist 3,South Division
125,Charles Robert Jackson,HR Specialist 3,South Division
126,Jessica Pamela Knight,HR Specialist 3,South Division
127,James Kyle Evans,HR Manager Level 3,South Division


**Q4.** Human Resources only: for each division, the number of HR employees and their average salary (2 decimals), highest average first.

In [17]:
# Q4
q('''
SELECT v.divtype, COUNT(e.emp_id), ROUND(AVG(e.salary), 2)
FROM public.employeedetails AS e
JOIN public.divisions AS v ON v.divisionid = e.divisionid
JOIN public.departments AS d ON d.departmentid = e.departmentid
WHERE d.departmentname = 'Human Resources'
GROUP BY v.divtype
ORDER BY AVG(e.salary) DESC;
''')

,divtype,count,round
0,East Division,129,53445.78
1,North Division,129,53045.74
2,Central Division,129,52765.92
3,West Division,129,52742.41
4,South Division,129,52523.17


**Q5.** Engineering employees whose salary is at or above the average Engineering salary: name, job title, salary. The average must be computed inside the query.

In [18]:
# Q5
q('''
SELECT p.name, t.jobtitle, e.salary
FROM public.personaldetails AS p
JOIN public.employeedetails AS e ON e.ssn = p.ssn
JOIN public.jobtitles AS t ON t.jobtitleid = e.jobtitleid
JOIN public.departments AS d ON d.departmentid = e.departmentid
WHERE d.departmentname = 'Engineering'
AND e.salary >= (
    SELECT AVG(emp.salary)
    FROM public.employeedetails AS emp
    JOIN public.departments AS dep ON dep.departmentid = emp.departmentid
    WHERE dep.departmentname = 'Engineering');
''')

,name,jobtitle,salary
0,Victoria Erica Lawson,Engineer Level 2,116951.610
1,Kyle Shaun Simmons,Engineer Level 3,121745.030
2,Jennifer Maria Oconnor,Engineer Level 2,116738.310
3,Hannah Katherine Dean,Engineer Level 5,111011.820
4,Joshua Matthew Bell,Engineering Manager Level 1,133120.880
...,...,...,...
456,Lisa Jennifer Gilbert,Engineering Manager Level 2,164438.888
457,Nicholas Ruben Morse,Engineer Level 3,121962.180
458,Alicia Pamela Williams,Engineer Level 4,117489.280
459,Devin Jeffrey Berry,Engineer Level 4,114721.720


**Q6.** Employees aged 50 or older versus under 50 (age as of today): for each group, the number of employees and the average salary. Leave out anyone with no birthdate.

In [19]:
# Q6
q('''
SELECT 
    CASE
        WHEN CURRENT_DATE - p.birthdate >= 50*365 THEN '>= 50'
        ELSE '< 50'
    END AS age,
    COUNT(e.emp_id), AVG(e.salary)
FROM public.employeedetails AS e
JOIN public.personaldetails AS p ON p.ssn = e.ssn
WHERE p.birthdate IS NOT NULL
GROUP BY
    CASE
        WHEN CURRENT_DATE - p.birthdate >= 50*365 THEN '>= 50'
        ELSE '< 50'
    END;
''')

,age,count,avg
0,>= 50,3273,46953.556846
1,< 50,11047,43845.878838


**Q7.** For each division: its city and state, its headcount and its average salary, largest headcount first.

In [20]:
# Q7
q('''
SELECT v.divtype, l.divcity, l.divstate, COUNT(e.emp_id), AVG(e.salary)
FROM public.employeedetails AS e
    JOIN public.divisions AS v ON v.divisionid = e.divisionid
    JOIN public.divisionlocations AS l ON l.divisionid = v.divisionid
GROUP BY v.divtype, l.divcity, l.divstate
ORDER BY COUNT(e.emp_id) DESC;
''')

,divtype,divcity,divstate,count,avg
0,South Division,Austin,TX,2868,45521.522943
1,West Division,Sacramento,CA,2863,44301.815015
2,North Division,Springfield,IL,2863,44323.953071
3,East Division,Albany,NY,2863,44215.797829
4,Central Division,Columbus,OH,2863,44416.096426


**Q8.** Departments whose average salary is higher than the company-wide average salary, with that department average.

In [21]:
# Q8
q('''
SELECT d.departmentname, AVG(e.salary)
FROM public.employeedetails AS e
    JOIN public.departments AS d ON d.departmentid = e.departmentid
GROUP BY d.departmentname
HAVING AVG(e.salary) > (SELECT AVG(salary) FROM public.employeedetails);
''')

,departmentname,avg
0,Management,227950.600000
1,Executive,756238.760000
2,Engineering,108286.633632
3,Human Resources,52904.603643


**Q9.** The highest-paid employee in each department: department, name, job title and salary.

In [22]:
# Q9
q('''
SELECT d.departmentname, p.name, t.jobtitle, e.salary
FROM public.personaldetails AS p
    JOIN public.employeedetails AS e ON e.ssn = p.ssn
    JOIN public.departments AS d ON d.departmentid = e.departmentid
    JOIN public.jobtitles AS t ON t.jobtitleid = e.jobtitleid
WHERE (d.departmentname, e.salary) IN (
    SELECT dep.departmentname, MAX(emp.salary)
    FROM public.employeedetails AS emp
        JOIN public.departments AS dep ON emp.departmentid = dep.departmentid
    GROUP BY dep.departmentname);   
''')

,departmentname,name,jobtitle,salary
0,Management,Darryl Taylor Heath,Division Manager,249261.08
1,Administrative,Christopher Justin Stanley,Secretary Level 8,55967.39
2,Human Resources,Michelle Angela Conley,Lead HR Manager,192030.36
3,Executive,Mark Alexander Hill,Chief Executive Officer,1111605.37
4,Operations,Veronica Rebecca Mclaughlin,Lead Operations Manager,293491.71
5,Engineering,Lindsey Breanna Villa,Lead Engineering Manager,291487.46


**Q10.** Data quality: every email address that belongs to more than one person, with how many people share it. Then answer in the markdown cell below.

In [23]:
# Q10
q('''
SELECT email, COUNT(email)
FROM public.personaldetails
GROUP BY email
HAVING COUNT(email) > 1;
''')

,email,count
0,anthony_thomas_200@KLB.com,2
1,luis_davis_161@KLB.com,2
2,michael_wilson_101@KLB.com,2
3,christopher_young_182@KLB.com,2
4,cheryl_peterson_139@KLB.com,2
5,james_smith_138@KLB.com,2
6,michael_morgan_159@KLB.com,2
7,michael_cook_148@KLB.com,2
8,joseph_davis_144@KLB.com,2
9,jessica_anderson_140@KLB.com,2


**Q10, part 2 (2 to 3 sentences):** Look at the rows Q10 returned. Is a shared email a data error, or could it be legitimate? KLB wants to use email as each employee's login. What would you check or change in the data or the schema before that is safe?


Some of them could be legitimate since the end of the cleaning process detected a couple of matching full names, but it's also possible that there was a typo in the numbers. The best way to verify if it is legitimate or a mistake is to compare some other values in each row like birthdate, social security number, employee ID, etc. to determine if an employee was duplicated for some reason. To use the emails as logins, a unique constraint would need to be added to the email column in the schema and duplicate emails would need to be handled (generate new unique email for erroneous entries, remove duplicate employee rows, etc) prior to loading.

# Part C: Dashboard

Your dashboard is `app.py`, a separate file. Build it from the specification in the instructions PDF, run it with `streamlit run app.py`, and put your screenshots and answers in the PDF. Nothing for Part C goes in this notebook.

# Part D: Industry scenarios

These are the kind of incidents data engineers handle at large companies. There is no single right answer. Answer each one in the same cell, under the question, in 3 to 5 sentences. Refer to **your own** loader, SQL or `app.py` wherever you can: a specific answer (the line, the query, the setting) beats a general one.

**D1. The job said SUCCESS.** Your loader runs every night. For months it has rejected a handful of rows each night. One Monday it rejects about 4,100 of 14,000 rows, the job still reports SUCCESS, and the dashboard's headcount drops by almost a third. Nobody notices for two days. The cause: the HR system started sending salaries as `"$85,000.00"` instead of `85000`.

1. Look at how **your** loader converts `Salary`. What would it do with `"$85,000.00"`, and where would those rows end up? If your loader would behave differently from the story above, say how.
2. Why did nothing fail?
3. What would you add to the loader so that next time this stops the run or alerts someone?


First, the loader would attempt to convert $85,000.00 to a number, which would fail given the non-numeric characters. When coercion fails, the loader inserts NaN which later causes the row to be placed in the rejects_employees.csv because salary can not be null in my schema. Moving a row to rejects does not currently raise any kind of error as it is meant to defend against errors happening during the actual loading of the data into the schema. If we wanted the system to alert someone of a rejected row, the system could log each time a row was placed into the rejects, produce a warning message, or even throw an error.

**D2. Two numbers for the same thing.** Finance's report and your dashboard both read the same `EmployeeDB`, but they show different average salaries for Engineering. Your manager asks which one is wrong.

1. Give three legitimate reasons the two numbers could differ without either query having a bug. At least one reason must come from how your own `app.py` works.
2. How would you settle which number the company should use?


One possibility is that the dashboard has sidebar filters applied that the finance report does not include, which disregards some rows and therefore changes the average. For this situation, the number the company should use would depend on whether those filters were actually intentional. If the filters are the same, the discrepency could be due to the fact that the  dashboard caches query results, so an update to the underlying data may not be reflected in the dashboard if that query has been run previously on old data. In this case, the company should use the finance report as it will reflect the current data. The third reason could be that the queries were run at different times and new data was inserted in between the two queries. The more accurate result in this situation would be whichever query was run later (assuming the dashboard query caching has been ruled out).

**D3. From 14,000 rows to 50 million.** After KLB acquires another company, the nightly file grows from about 14,000 rows to 50 million.

1. Time one run of your loader on your own data (for example, wrap one of the runs in A4 with `time.perf_counter()`) and report the time.
2. What in your loader would break or slow down first at 50 million rows? Point to the specific part of your code and explain why it gets slow.
3. What would you change?


The current run time of the loader is 3.9 seconds. The aspect of the loader that would break or slow down the most at 50 million rows is the checks involving `SELECT ... WHERE NOT EXISTS ...`, especially for the employeedetails table, as it has to check every row already inserted to check for duplicates. When there are 50 million rows, this becomes very time consuming and could cause a query to time out. To address this issue, I would add an index on the relevant columns that are being used to identify duplicate entires (e.g. ssn and employeeid) to improve lookup time.